In [2]:

import csv, json
from pathlib import Path

In [5]:
"""
Day 01: Functions, type hints, exceptions, and file handling.

Loads skin-lesion screening cases from a CSV file, validates each row,
summarises the valid cases, and saves a JSON report.
"""

import csv
import json
from pathlib import Path

# A "type alias": a short name for a long type, so signatures stay readable.
Case = dict[str, str | int | float]

VALID_DIAGNOSES: set[str] = {
    "buruli_ulcer", "leprosy", "scabies", "yaws", "mycetoma", "normal",
}


class InvalidCaseError(ValueError):
    """Raised when a row in the case file breaks a validation rule."""


def create_sample_csv(path: Path) -> None:
    """Write a small demo dataset, including some deliberately bad rows."""
    rows = [
        ["case_id", "age", "sex", "diagnosis", "lesion_size_mm"],
        ["C001", "34", "F", "buruli_ulcer", "22.5"],
        ["C002", "8", "M", "scabies", "3.0"],
        ["C003", "abc", "F", "yaws", "5.5"],
        ["C004", "51", "M", "leprosy", "15.0"],
        ["C005", "27", "F", "malaria", "0"],
        ["C006", "45", "M", "mycetoma", "-4"],
    ]
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerows(rows)


def parse_case(row: dict[str, str]) -> Case:
    """Convert one raw CSV row (all strings) into a validated case."""
    case_id = row["case_id"].strip()

    try:
        age = int(row["age"])
        lesion_size = float(row["lesion_size_mm"])
    except ValueError as exc:
        raise InvalidCaseError(f"{case_id}: age/size must be numbers ({exc})") from exc

    if not 0 <= age <= 120:
        raise InvalidCaseError(f"{case_id}: age {age} is outside 0-120")
    if lesion_size < 0:
        raise InvalidCaseError(f"{case_id}: lesion size cannot be negative")

    diagnosis = row["diagnosis"].strip().lower()
    if diagnosis not in VALID_DIAGNOSES:
        raise InvalidCaseError(f"{case_id}: unknown diagnosis '{diagnosis}'")

    return {
        "case_id": case_id,
        "age": age,
        "sex": row["sex"].strip().upper(),
        "diagnosis": diagnosis,
        "lesion_size_mm": lesion_size,
    }


def load_cases(path: Path) -> tuple[list[Case], list[str]]:
    """Read the CSV and split rows into valid cases and error messages."""
    valid_cases: list[Case] = []
    errors: list[str] = []

    with path.open("r", newline="", encoding="utf-8") as file:
        reader = csv.DictReader(file)
        for line_number, row in enumerate(reader, start=2):
            try:
                valid_cases.append(parse_case(row))
            except InvalidCaseError as exc:
                errors.append(f"line {line_number}: {exc}")

    return valid_cases, errors


def summarise(cases: list[Case]) -> dict[str, object]:
    """Count cases per diagnosis and compute the mean age."""
    if not cases:
        return {"total_cases": 0, "by_diagnosis": {}, "mean_age": None}

    counts: dict[str, int] = {}
    for case in cases:
        diagnosis = str(case["diagnosis"])
        counts[diagnosis] = counts.get(diagnosis, 0) + 1

    mean_age = sum(int(case["age"]) for case in cases) / len(cases)

    return {
        "total_cases": len(cases),
        "by_diagnosis": counts,
        "mean_age": round(mean_age, 1),
    }


def save_report(report: dict[str, object], path: Path) -> None:
    """Save the report dictionary as pretty-printed JSON."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as file:
        json.dump(report, file, indent=2)


def main() -> None:
    data_path = Path("data") / "cases.csv"
    report_path = Path("outputs") / "report.json"

    if not data_path.exists():
        create_sample_csv(data_path)
        print(f"Created sample data at {data_path}")

    try:
        cases, errors = load_cases(data_path)
    except OSError as exc:
        print(f"Could not read {data_path}: {exc}")
        return

    report = summarise(cases)
    report["rejected_rows"] = errors
    save_report(report, report_path)

    print(f"Valid cases: {len(cases)} | Rejected: {len(errors)}")
    for error in errors:
        print(f"  - {error}")
    print(f"Report saved to {report_path}")


if __name__ == "__main__":
    main()

Valid cases: 3 | Rejected: 3
  - line 4: COO3: age/size must be numbers (invalid literal for int() with base 10: 'abc')
  - line 6: COO5: unknown diagnosis 'malaria'
  - line 7: COO6: lesion size cannot be negative
Report saved to outputs\report.json
